### Monthly order summary
#### For each of the customer, produce the following summary per month
##### 1. total orders
##### 2. total items bought
##### 3. total amount spent

In [0]:
df_orders = spark.read.table('gizmobox_catalog_noori.silver.orders_details')
display(df_orders)

In [0]:

from pyspark.sql import functions as f
df_monthly_summary = (df_orders                   
                      .groupBy([ f.date_format("transaction_timestamp", "yyyy-MM").alias('order_month'),'customer_id'])
                      .agg(                      
                     
                         f.countDistinct('order_id').alias('total_orders'),
                         f.sum('item_price').alias('total_amount'),
                         f.sum('item_quantity').alias('total_quantity'),
                         f.sum(f.col("item_price") * f.col("item_quantity")).alias('total_revenue')
                        )
                      
                      )

                         
display(df_monthly_summary
)

In [0]:
from pyspark.sql import functions as f

df_monthly_summary = (
    df_orders
    .select(
        f.date_format("transaction_timestamp", "yyyy-MM").alias("order_month"),
        "customer_id",
        "order_id",
        "item_price",
        "item_quantity"
    )
    .groupBy( "order_month","customer_id")
    .agg(
        
        f.countDistinct("order_id").alias("total_orders"),
        f.sum("item_price").alias("total_amount"),
        f.sum("item_quantity").alias("total_quantity"),
        f.sum(f.col("item_price") * f.col("item_quantity")).alias("total_revenue")
    )
)

display(df_monthly_summary)

In [0]:
from pyspark.sql import functions as f

df_monthly_summary = (
    df_orders
    .withColumn(
        "order_month",
        f.date_format("transaction_timestamp", "yyyy-MM")
    )
    .groupBy("customer_id", "order_month")
    .agg(
        f.countDistinct("order_id").alias("total_orders"),
        f.sum("item_price").alias("total_amount"),
        f.sum("item_quantity").alias("total_quantity"),
        f.sum(f.col("item_price") * f.col("item_quantity")).alias("total_revenue")
    )
)

display(df_monthly_summary)

In [0]:
df_monthly_summary.writeTo('gizmobox_catalog_noori.gold.py1_order_summary_montly').createOrReplace()

In [0]:
spark.read.table('gizmobox_catalog_noori.gold.py1_order_summary_montly').display()